# Fase 1 · Paso 2: inventario de adquisiciones basales

Esta libreta valida las 1,916 adquisiciones esperadas, audita únicamente los paquetes ya descargados y genera la cola privada para descarga selectiva. No entrena modelos ni abre la prueba reservada.

## 1. Montar Drive y localizar la raíz privada

In [ ]:
import os
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
private_root = userdata.get('KNEE_DATA_ROOT') or os.environ.get('KNEE_DATA_ROOT')
if not private_root:
    raise RuntimeError('Configura el secreto KNEE_DATA_ROOT antes de continuar.')
os.environ['KNEE_DATA_ROOT'] = private_root
data_root = Path(private_root)
if not data_root.is_dir():
    raise FileNotFoundError('La raíz privada configurada no está disponible.')
(data_root / 'dicom' / 'originales' / 'cohorte_v00').mkdir(parents=True, exist_ok=True)
print('Drive montado y estructura privada preparada.')

## 2. Obtener una revisión identificable del repositorio

In [ ]:
import shutil
import subprocess

REPO_URL = 'https://github.com/AlonsoYess/Knee.git'
BRANCH = 'codex/fase-0-base-reproducible'
REPO_DIR = Path('/content/Knee_fase1_paso2')

if REPO_DIR.exists():
    if not (REPO_DIR / '.git').is_dir():
        raise RuntimeError('La ruta de trabajo existe y no es un clon Git; usa un entorno nuevo.')
    dirty = subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout
    if dirty:
        raise RuntimeError('El clon temporal contiene cambios. Restablece el entorno de ejecución.')
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'merge', '--ff-only', f'origin/{BRANCH}'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
print('Revisión preparada:', commit)

## 3. Instalar y comprobar el código

In [ ]:
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=REPO_DIR, check=True)
if subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout:
    raise RuntimeError('Las comprobaciones modificaron el repositorio temporal.')
print('Pruebas automáticas completadas.')

## 4. Construir el inventario y la cola de descarga selectiva

Los identificadores, rutas y huellas se guardan solo en Drive. La salida mostrada por pantalla contiene únicamente recuentos agregados.

In [ ]:
config_path = REPO_DIR / 'configs' / 'acquisition_inventory.example.json'
subprocess.run([sys.executable, '-m', 'knee.acquisition_inventory', '--config', str(config_path)], cwd=REPO_DIR, check=True)

## 5. Verificar y registrar la ejecución

In [ ]:
import json
from datetime import datetime, timezone

output_dir = data_root / 'outputs' / 'auditorias' / 'fase_1' / 'paso_2_inventario'
summary_path = output_dir / 'resumen_inventario_publico.json'
inventory_path = output_dir / 'inventario_adquisiciones_privado.csv'
queue_path = output_dir / 'cola_descarga_selectiva_privada.csv'
for required in (summary_path, inventory_path, queue_path):
    if not required.is_file():
        raise FileNotFoundError(f'No se generó la evidencia esperada: {required.name}')
summary = json.loads(summary_path.read_text(encoding='utf-8'))
if summary['manifest_contract']['status'] != 'ok':
    raise RuntimeError('El contrato del manifiesto no fue validado.')
if summary['expected_unique_acquisitions'] != 1916:
    raise RuntimeError('El inventario no conserva las 1,916 adquisiciones esperadas.')
record = {
    'phase': 1,
    'step': 2,
    'executed_at_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': commit,
    'inventory_status': summary['status'],
    'expected_unique_acquisitions': summary['expected_unique_acquisitions'],
    'local_status_counts': summary['local_status_counts'],
    'pending_download_or_replacement': summary['pending_download_or_replacement'],
    'download_batches': summary['download_batches'],
    'training_executed': False,
    'reserved_test_opened': False,
}
(output_dir / 'registro_ejecucion_fase1_paso2.json').write_text(json.dumps(record, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
print('Fase 1, paso 2 preparado. Inventario y cola privada guardados en Drive.', record)